# Limpieza, deduplicacion y creacion de chunks

Objetivo: convertir transcripciones crudas en fragmentos etiquetables, conservando trazabilidad hacia video, canal, tiempo de inicio y hash de texto.

In [1]:
!pip3 install -q pandas scikit-learn tqdm

In [2]:
from pathlib import Path
import hashlib
import json
import re
import unicodedata

import pandas as pd

ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'datos' / 'raw'
INTERIM_DIR = ROOT / 'datos' / 'interim'
PROCESSED_DIR = ROOT / 'datos' / 'processed'

for path in [INTERIM_DIR, PROCESSED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RAW_TRANSCRIPTS = RAW_DIR / 'transcripts_raw.jsonl'
print('Entrada esperada:', RAW_TRANSCRIPTS)

Entrada esperada: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\raw\transcripts_raw.jsonl


## 1. Taxonomia de etiquetado — contextualizada en Peru

El etiquetado es **multi-etiqueta**: un chunk puede activar varias subcategorias simultaneamente.
Las categorias principales son no mutuamente excluyentes salvo `seguro` vs. etiquetas de dano.

Fundamento academico peruano:
- **Racismo negado** → Portocarrero (2009); Vich (2018): el racismo se niega y se presenta como criterio educativo.
- **Racismo linguistico** → Almeida & Zavala (2022): burla del motoseo, acento andino y ortografia de migrantes.
- **Clasismo-racismo entrelazado** → Callirgos (1993); Branez Medina (2012): raza y clase son inseparables en Peru.
- **Humor como cobertura** → Branez Medina (2012): el dano puede ser real aunque el tono sea jocoso.
- **Misoginia digital** → Monge-Olivarria & Guerra-Corrales (2023): feminizacion como insulto en redes peruanas.
- **Falla en idiomas locales** → Thakur / CDT (2025): sistemas automaticos fallan con espanol andino y quechua.

Los **flags transversales** no son categorias de dano sino indicadores de ambiguedad que el moderador humano debe resolver.

In [3]:
TAXONOMY = pd.DataFrame([
    # ── Seguro ────────────────────────────────────────────────────────────────
    {'label': 'seguro',
     'categoria': 'SEGURO',
     'descripcion': 'Sin infraccion. Texto informativo, descriptivo o humor sin ataque a persona o grupo.',
     'fuente': ''},
    {'label': 'seguro_ironia_marcada',
     'categoria': 'SEGURO',
     'descripcion': 'Parodia o ironia cuyo blanco NO es un grupo humano. El anotador confirma ausencia de dano.',
     'fuente': ''},
    # ── Racismo / Discriminacion ───────────────────────────────────────────────
    {'label': 'racismo_etnico_explicito',
     'categoria': 'RACISMO_DISCRIMINACION',
     'descripcion': 'Uso derogatorio de serrano, cholo, negro, indio u otros terminos etnicos.',
     'fuente': 'Callirgos (1993); Zavala & Back (2017)'},
    {'label': 'racismo_linguistico',
     'categoria': 'RACISMO_DISCRIMINACION',
     'descripcion': 'Burla del acento andino, el motoseo o la ortografia de migrantes (amixer).',
     'fuente': 'Almeida & Zavala (2022)'},
    {'label': 'clasismo_racial',
     'categoria': 'RACISMO_DISCRIMINACION',
     'descripcion': 'Inferioriza por clase social con connotacion etnica (huachafa, chusma, amixer).',
     'fuente': 'Callirgos (1993); Branez Medina (2012)'},
    {'label': 'discriminacion_regional',
     'categoria': 'RACISMO_DISCRIMINACION',
     'descripcion': 'Ataque por ser provinciano, serrano o de fuera de Lima.',
     'fuente': 'Zavala & Zariquiey (2007)'},
    {'label': 'racismo_encubierto',
     'categoria': 'RACISMO_DISCRIMINACION',
     'descripcion': 'Discriminacion disfrazada de criterio de educacion o cultura (fundamento invisible).',
     'fuente': 'Zavala & Zariquiey (2007); Portocarrero (2009)'},
    # ── Acoso ─────────────────────────────────────────────────────────────────
    {'label': 'misoginia_acoso_genero',
     'categoria': 'ACOSO',
     'descripcion': 'Insultos sexualizados, degradacion o ataque por ser mujer; feminizacion como insulto.',
     'fuente': 'Monge-Olivarria & Guerra-Corrales (2023)'},
    {'label': 'homofobia_transfobia',
     'categoria': 'ACOSO',
     'descripcion': 'Insultos o amenazas contra personas LGBTQ+ por orientacion o identidad.',
     'fuente': ''},
    {'label': 'acoso_personal',
     'categoria': 'ACOSO',
     'descripcion': 'Ataque dirigido a persona identificable por nombre, cargo o rol; doxeo.',
     'fuente': ''},
    {'label': 'amenaza_directa',
     'categoria': 'ACOSO',
     'descripcion': 'Expresion explicita de intencion de dano fisico, legal o economico.',
     'fuente': ''},
    # ── Contenido sexual ──────────────────────────────────────────────────────
    {'label': 'sexual_explicito',
     'categoria': 'CONTENIDO_SEXUAL',
     'descripcion': 'Descripcion grafica de actos sexuales sin proposito informativo.',
     'fuente': ''},
    {'label': 'sexual_cosificacion',
     'categoria': 'CONTENIDO_SEXUAL',
     'descripcion': 'Sexualizacion o cosificacion de personas como objetos sexuales.',
     'fuente': ''},
    {'label': 'sexual_no_consensual',
     'categoria': 'CONTENIDO_SEXUAL',
     'descripcion': 'Referencia a contenido no consensual o revenge porn.',
     'fuente': ''},
    # ── Flags transversales (coexisten con categorias de dano) ────────────────
    {'label': 'ironia_ambigua',
     'categoria': 'FLAG',
     'descripcion': 'No se determina si el dano es intencional o parodia critica. Activa revision humana.',
     'fuente': 'Vich (2018); Branez Medina (2012)'},
    {'label': 'humor_encubridor',
     'categoria': 'FLAG',
     'descripcion': 'Humor usado para negar el dano (es broma). El dano puede ser real encubierto.',
     'fuente': 'Branez Medina (2012)'},
    {'label': 'contexto_necesario',
     'categoria': 'FLAG',
     'descripcion': 'Chunk aislado insuficiente para clasificar. Requiere ver el video completo.',
     'fuente': 'Thakur / CDT (2025)'},
])

TAXONOMY.to_csv(PROCESSED_DIR / 'taxonomia_moderacion.csv', index=False)

resumen = TAXONOMY.groupby('categoria')['label'].apply(list)
print('Etiquetas por categoria:')
for cat, labels in resumen.items():
    print(f'  {cat} ({len(labels)}): {", ".join(labels)}')
TAXONOMY


Etiquetas por categoria:
  ACOSO (4): misoginia_acoso_genero, homofobia_transfobia, acoso_personal, amenaza_directa
  CONTENIDO_SEXUAL (3): sexual_explicito, sexual_cosificacion, sexual_no_consensual
  FLAG (3): ironia_ambigua, humor_encubridor, contexto_necesario
  RACISMO_DISCRIMINACION (5): racismo_etnico_explicito, racismo_linguistico, clasismo_racial, discriminacion_regional, racismo_encubierto
  SEGURO (2): seguro, seguro_ironia_marcada


,label,categoria,descripcion,fuente
0,seguro,SEGURO,"Sin infraccion. Texto informativo, descriptivo...",
1,seguro_ironia_marcada,SEGURO,Parodia o ironia cuyo blanco NO es un grupo hu...,
2,racismo_etnico_explicito,RACISMO_DISCRIMINACION,"Uso derogatorio de serrano, cholo, negro, indi...",Callirgos (1993); Zavala & Back (2017)
3,racismo_linguistico,RACISMO_DISCRIMINACION,"Burla del acento andino, el motoseo o la ortog...",Almeida & Zavala (2022)
4,clasismo_racial,RACISMO_DISCRIMINACION,Inferioriza por clase social con connotacion e...,Callirgos (1993); Branez Medina (2012)
5,discriminacion_regional,RACISMO_DISCRIMINACION,"Ataque por ser provinciano, serrano o de fuera...",Zavala & Zariquiey (2007)
6,racismo_encubierto,RACISMO_DISCRIMINACION,Discriminacion disfrazada de criterio de educa...,Zavala & Zariquiey (2007); Portocarrero (2009)
7,misoginia_acoso_genero,ACOSO,"Insultos sexualizados, degradacion o ataque po...",Monge-Olivarria & Guerra-Corrales (2023)
8,homofobia_transfobia,ACOSO,Insultos o amenazas contra personas LGBTQ+ por...,
9,acoso_personal,ACOSO,Ataque dirigido a persona identificable por no...,


In [4]:
def load_jsonl(path):
    if not path.exists():
        print('No existe el archivo:', path)
        return []
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def normalize_text(text):
    text = unicodedata.normalize('NFKC', text or '')
    text = text.replace('\n', ' ')
    text = re.sub(r'\[(musica|aplausos|risas|music|applause|laughter)\]', ' ', text, flags=re.I)
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def text_hash(text):
    return hashlib.md5(normalize_text(text).lower().encode('utf-8')).hexdigest()


def remove_vtt_overlap(prev_text, next_text, max_words=12):
    """Elimina el prefijo de next_text que ya aparece como sufijo de prev_text.

    Los archivos .vtt usan *rolling captions*: cada bloque nuevo repite las
    últimas N palabras del bloque anterior para facilitar la lectura humana.
    Sin esta corrección, al concatenar segmentos se producen duplicados:

        seg1 = "el presidente anunció hoy"
        seg2 = "anunció hoy nuevas medidas"   ← "anunció hoy" ya está en seg1
        resultado sin corrección → "el presidente anunció hoy anunció hoy nuevas medidas"
        resultado con corrección → "el presidente anunció hoy nuevas medidas"

    Algoritmo: busca el solapamiento más largo entre el sufijo de prev_text
    y el prefijo de next_text, y elimina ese prefijo de next_text.

    Args:
        prev_text : último segmento acumulado en el chunk (ya normalizado).
        next_text : texto del nuevo segmento a agregar (ya normalizado).
        max_words : ventana máxima de palabras a comparar como solapamiento.
    Returns:
        next_text sin el prefijo solapado; cadena vacía si todo era solapamiento.
    """
    if not prev_text or not next_text:
        return next_text

    prev_words  = prev_text.lower().split()
    next_lower  = next_text.lower().split()
    next_orig   = next_text.split()

    window = min(max_words, len(prev_words), len(next_lower))
    for overlap in range(window, 0, -1):
        if prev_words[-overlap:] == next_lower[:overlap]:
            remaining = next_orig[overlap:]
            return ' '.join(remaining)  # puede ser '' si todo era solapamiento

    return next_text


In [8]:
def build_chunks(record, target_seconds=30, max_chars=600, min_chars=90):
    """Construye chunks de texto a partir de los segmentos de una transcripción.

    Antes de acumular cada segmento elimina el solapamiento VTT (rolling captions)
    con el segmento anterior mediante remove_vtt_overlap.
    """
    chunks = []
    current = []
    start = None
    end = None
    char_count = 0

    for seg in record.get('segments', []):
        seg_text = normalize_text(seg.get('text', ''))
        if not seg_text:
            continue

        # ── Corrección de solapamiento VTT ───────────────────────────────────
        # Los .vtt repiten palabras al final/inicio de bloques consecutivos.
        # Eliminamos el prefijo de seg_text que ya está al final del último segmento.
        if current:
            seg_text = remove_vtt_overlap(current[-1], seg_text)
            if not seg_text:        # todo era solapamiento → descartar este segmento
                continue

        seg_start = float(seg.get('start', 0.0))
        seg_end   = seg_start + float(seg.get('duration', 0.0))
        if start is None:
            start = seg_start
        end = seg_end
        current.append(seg_text)
        char_count += len(seg_text)

        if (end - start >= target_seconds) or (char_count >= max_chars):
            text = normalize_text(' '.join(current))
            if len(text) >= min_chars:
                chunks.append(make_chunk(record, start, end, text, len(chunks)))
            current, start, end, char_count = [], None, None, 0

    # último chunk parcial
    text = normalize_text(' '.join(current))
    if len(text) >= min_chars:
        chunks.append(make_chunk(record, start, end, text, len(chunks)))
    return chunks


def make_chunk(record, start, end, text, idx):
    video_id = record.get('video_id', 'sin_video')
    chunk_id  = f'{video_id}_{idx:04d}'
    return {
        'chunk_id':      chunk_id,
        'video_id':      video_id,
        'channel_id':    record.get('channel_id'),
        'channel_title': record.get('channel_title'),
        'video_title':   record.get('title'),
        'published_at':  record.get('published_at'),
        'start_seconds': round(float(start or 0.0), 2),
        'end_seconds':   round(float(end or 0.0), 2),
        'text':          text,
        'text_hash':     text_hash(text),
        'labels':        [],   # etiquetas de categoría (RACISMO, ACOSO, SEXUAL, SEGURO…)
        'flags':         [],   # flags transversales (ironia_ambigua, humor_encubridor, …)
        'needs_review':  True,
        'annotator':     '',
        'notes':         '',
    }


In [9]:
records = load_jsonl(RAW_TRANSCRIPTS)
chunks = []
for record in records:
    chunks.extend(build_chunks(record))

chunks_df = pd.DataFrame(chunks)
if not chunks_df.empty:
    chunks_df = chunks_df.drop_duplicates('text_hash').reset_index(drop=True)
    chunks_df.to_csv(PROCESSED_DIR / 'chunks_para_etiquetar.csv', index=False)
    write_jsonl(chunks_df.to_dict(orient='records'), PROCESSED_DIR / 'chunks_para_etiquetar.jsonl')

print('Registros fuente:', len(records))
print('Chunks unicos:', len(chunks_df))
chunks_df.head()

Registros fuente: 317
Chunks unicos: 15150


,chunk_id,video_id,channel_id,channel_title,video_title,published_at,start_seconds,end_seconds,text,text_hash,labels,flags,needs_review,annotator,notes
0,doBMCJT5Y58_0000,doBMCJT5Y58,None,El diario de Curwen,PRIDE DAY ON POLITICAL BRUTALITY | #POLITICALB...,None,108.27,139.23,"Buenas noches. Muy buenas noches. No, no se va...",da652f86791368414191595609427814,[],[],True,,
1,doBMCJT5Y58_0001,doBMCJT5Y58,None,El diario de Curwen,PRIDE DAY ON POLITICAL BRUTALITY | #POLITICALB...,None,139.23,169.87,"Víctor Caballero, yo no lo soy, yo soy mostace...",ff5c48ab67aa7eafd4eee3e29f6dd862,[],[],True,,
2,doBMCJT5Y58_0002,doBMCJT5Y58,None,El diario de Curwen,PRIDE DAY ON POLITICAL BRUTALITY | #POLITICALB...,None,169.87,202.51,conocimientos y se van a divertir. Les prometo...,cb5d4020b31ada3ef108a6e74ad01bd8,[],[],True,,
3,doBMCJT5Y58_0003,doBMCJT5Y58,None,El diario de Curwen,PRIDE DAY ON POLITICAL BRUTALITY | #POLITICALB...,None,202.51,232.51,"buenos, buenas, lo que sea, estoy feliz de est...",784eac29069e3fba90dfa34fa4e40384,[],[],True,,
4,doBMCJT5Y58_0004,doBMCJT5Y58,None,El diario de Curwen,PRIDE DAY ON POLITICAL BRUTALITY | #POLITICALB...,None,232.51,264.47,una importantísima figura de las redes sociale...,b26a9b6a0d79fb71549f0d78fc495ebe,[],[],True,,


## 2. Checklist de calidad

- Cada chunk conserva `video_id`, `channel_title`, `start_seconds`, `end_seconds` y `chunk_id`.
- **Solapamiento VTT (rolling captions)**: `remove_vtt_overlap` elimina las palabras repetidas
  al inicio de cada nuevo segmento antes de acumularlo. Sin esta corrección, una palabra presente
  al final del segmento i y al inicio del segmento i+1 aparecería dos veces en el chunk.
- Los duplicados de chunks completos se eliminan por `text_hash` tras normalizar texto.
- El campo `flags` es independiente de `labels`; ambos deben estar presentes en el JSONL de salida.
- `seguro` y `seguro_ironia_marcada` no pueden coexistir con etiquetas de daño.
- Chunks con `contexto_necesario` activo se marcan para revisión humana; no se descartan automáticamente.